In [20]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

In [ ]:
'''def get_binance_klines(symbol='BTCUSDT', interval='1h', limit=1000):
    """
    获取 Binance 交易对的历史 K 线数据。
    
    参数:
    - symbol: 交易对（例如 BTCUSDT）
    - interval: 时间间隔（例如 '1m', '5m', '1h', '1d'）
    - limit: 返回数据的数量限制（最多 1000）
    
    返回:
    - DataFrame 包含时间、开盘价、最高价、最低价、收盘价、交易量。
    """
    url = "https://api.binance.com/api/v3/klines"
    params = {
        'symbol': symbol,
        'interval': interval,
        'limit': limit
    }
    
    response = requests.get(url, params=params)
    data = response.json()

    # 创建 DataFrame
    columns = ['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close Time', 
               'Quote Asset Volume', 'Number of Trades', 'Taker Buy Base Volume', 
               'Taker Buy Quote Volume', 'Ignore']
    df = pd.DataFrame(data, columns=columns)

    # 转换时间戳
    df['Open Time'] = pd.to_datetime(df['Open Time'], unit='ms')
    df['Close Time'] = pd.to_datetime(df['Close Time'], unit='ms')

#return df
# 精简字段返回
    return df[['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume']]

# 示例用法
df_binance = get_binance_klines(symbol='BTCUSDT', interval='1h', limit=1000)'''

## 循环获取多段

In [21]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

def get_binance_klines_range(symbol='BTCUSDT', interval='1h', start_str=None, end_str=None):
    """
    获取指定时间区间内的 Binance 历史 K 线数据（自动循环获取多批次）
    """

    # 转换时间为毫秒时间戳
    start_ts = int(pd.to_datetime(start_str).timestamp() * 1000) if start_str else None
    end_ts = int(pd.to_datetime(end_str).timestamp() * 1000) if end_str else None

    # Binance API 限制每次最多返回 1000 条
    limit = 1000
    url = "https://api.binance.com/api/v3/klines"
    all_data = []

    while True:
        params = {
            'symbol': symbol,
            'interval': interval,
            'limit': limit
        }
        if start_ts:
            params['startTime'] = start_ts
        if end_ts:
            params['endTime'] = end_ts

        response = requests.get(url, params=params)
        data = response.json()

        # ✅ 检查返回是否为错误信息
        if not isinstance(data, list):
            print("⚠️ API 返回错误：", data)
            break

        if not data:
            print("✅ 数据获取完毕。")
            break

        all_data.extend(data)

        # ✅ 取最后一条的关闭时间 + 1ms 作为下次的起始时间
        last_close_time = data[-1][6]
        start_ts = last_close_time + 1

        # ✅ 如果超过结束时间则停止
        if end_ts and start_ts >= end_ts:
            break

        # 控制请求频率
        time.sleep(0.2)

    # 创建 DataFrame
    columns = ['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close Time',
               'Quote Asset Volume', 'Number of Trades', 'Taker Buy Base Volume',
               'Taker Buy Quote Volume', 'Ignore']
    df = pd.DataFrame(all_data, columns=columns)

    # 转换时间格式
    df['Open Time'] = pd.to_datetime(df['Open Time'], unit='ms')
    df['Close Time'] = pd.to_datetime(df['Close Time'], unit='ms')

    # 转为数值型
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    df[numeric_cols] = df[numeric_cols].astype(float)

    return df

In [22]:
#指定时间段数据
df = get_binance_klines_range(
    symbol='BTCUSDT',
    interval='1h',
    start_str='2025-04-27 00:00:00',
    end_str='2025-10-27 08:00:00'
)

In [27]:
#指定时间段数据
df_ETH = get_binance_klines_range(
    symbol='ETHUSDT',
    interval='1h',
    start_str='2025-04-27 00:00:00',
    end_str='2025-10-27 08:00:00'
)

In [16]:
#指定时间前1000个小时
# 先设置结束时间
end_time = '2025-09-15 17:00:00'
# 然后往前取1000小时
start_time = (pd.to_datetime(end_time) - timedelta(hours=1000)).strftime('%Y-%m-%d %H:%M:%S')

df = get_binance_klines_range('BTCUSDT', '1h', start_time, end_time)
print(df.shape)

KeyError: -1

In [28]:
df_ETH

,Open Time,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume,Ignore
0,2025-04-27 00:00:00,1820.87,1849.50,1820.00,1842.03,33911.0313,2025-04-27 00:59:59.999,62399752.23837700,240534,20118.92050000,37015481.36426200,0
1,2025-04-27 01:00:00,1842.04,1857.47,1801.02,1804.60,63309.4394,2025-04-27 01:59:59.999,115917894.03091300,420760,30431.50770000,55803449.49551000,0
2,2025-04-27 02:00:00,1804.61,1818.10,1801.07,1813.24,14113.3224,2025-04-27 02:59:59.999,25575684.70299800,191806,6440.36080000,11671173.70575300,0
3,2025-04-27 03:00:00,1813.23,1817.50,1805.55,1809.75,14442.8477,2025-04-27 03:59:59.999,26166752.24177500,146442,5378.47510000,9741535.16370900,0
4,2025-04-27 04:00:00,1809.75,1811.60,1797.02,1798.20,15348.8644,2025-04-27 04:59:59.999,27682605.10238300,97762,5475.48690000,9876673.47388200,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4396,2025-10-27 04:00:00,4205.08,4217.57,4193.24,4212.63,15008.4733,2025-10-27 04:59:59.999,63071527.21015900,123172,5223.86620000,21962204.55082300,0
4397,2025-10-27 05:00:00,4212.62,4253.72,4210.05,4233.23,26844.6284,2025-10-27 05:59:59.999,113704642.23409500,213486,14359.31000000,60816920.56673600,0
4398,2025-10-27 06:00:00,4233.22,4245.06,4222.80,4235.21,14206.4403,2025-10-27 06:59:59.999,60130496.13341100,173060,6126.54220000,25930441.69425700,0
4399,2025-10-27 07:00:00,4235.21,4247.56,4196.05,4201.36,22342.3093,2025-10-27 07:59:59.999,94199216.68604400,218717,8931.85780000,37655523.38824100,0


In [29]:
df_ETH.to_csv("Binance_ETH_hour.csv")